In [3]:
# Setup
!pip install spacy
!python -m spacy download en_core_web_sm

import nltk
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('punkt_tab')

import spacy
import pandas as pd
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.tree import Tree

print("Week 3 libraries loaded ")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 64.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Week 3 libraries loaded 


[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Packag

In [2]:
# Dataset  for Training / Testing

court_documents = [
    {
        "case_id"  : "CASE-001",
        "text"     : "Person A filed a suit against Organization X in the High Court at City A on 14th March 2023. Judge B presided over the case.",
        "type"     : "Civil"
    },
    {
        "case_id"  : "CASE-002",
        "text"     : "Person C was convicted by Judge D at Court B on 5th January 2023 for fraud against Organization Y.",
        "type"     : "Criminal"
    },
    {
        "case_id"  : "CASE-003",
        "text"     : "Organization Z appealed against the ruling by Judge E at the Court of Appeal in City B on 20th April 2023.",
        "type"     : "Commercial"
    },
    {
        "case_id"  : "CASE-004",
        "text"     : "The County Government of Region A was ordered by Judge F to compensate Person D for unlawful land acquisition in Town A on 8th February 2023.",
        "type"     : "Land"
    },
    {
        "case_id"  : "CASE-005",
        "text"     : "Person E filed a petition against the Electoral Commission at the Supreme Court in City A on 2nd August 2022.",
        "type"     : "Constitutional"
    },
]

print("COURT DOCUMENT DATASET")
print("=" * 65)
print(f"{'Case ID':<12} {'Type':<16} {'Text Preview'}")
print("-" * 65)
for doc in court_documents:
    print(f"{doc['case_id']:<12} {doc['type']:<16} {doc['text'][:40]}...")

print(f"\nTotal documents: {len(court_documents)}")

COURT DOCUMENT DATASET
Case ID      Type             Text Preview
-----------------------------------------------------------------
CASE-001     Civil            Person A filed a suit against Organizati...
CASE-002     Criminal         Person C was convicted by Judge D at Cou...
CASE-003     Commercial       Organization Z appealed against the ruli...
CASE-004     Land             The County Government of Region A was or...
CASE-005     Constitutional   Person E filed a petition against the El...

Total documents: 5


In [4]:
# Sequence Labeling Code using SpaCy

nlp = spacy.load("en_core_web_sm")

label_map = {
    'PERSON'  : 'Person Name',
    'ORG'     : 'Organization',
    'GPE'     : 'Location / City',
    'DATE'    : 'Date',
    'LAW'     : 'Law or Regulation',
    'MONEY'   : 'Money / Amount',
    'CARDINAL': 'Number',
}

print("SEQUENCE LABELING OUTPUT (SpaCy NER)")
print("=" * 65)

for doc_data in court_documents:
    doc = nlp(doc_data['text'])
    print(f"\nCase: {doc_data['case_id']}  |  Type: {doc_data['type']}")
    print(f"Text: {doc_data['text'][:60]}...")
    print(f"  {'Entity Found':<30} {'Label':<12} {'What it means'}")
    print(f"  {'-'*55}")
    if doc.ents:
        for ent in doc.ents:
            meaning = label_map.get(ent.label_, ent.label_)
            print(f"  {ent.text:<30} {ent.label_:<12} {meaning}")
    else:
        print("  No entities found.")

SEQUENCE LABELING OUTPUT (SpaCy NER)

Case: CASE-001  |  Type: Civil
Text: Person A filed a suit against Organization X in the High Cou...
  Entity Found                   Label        What it means
  -------------------------------------------------------
  Organization X in the High Court ORG          Organization
  14th March 2023                DATE         Date

Case: CASE-002  |  Type: Criminal
Text: Person C was convicted by Judge D at Court B on 5th January ...
  Entity Found                   Label        What it means
  -------------------------------------------------------
  5th January 2023               DATE         Date
  Organization Y.                ORG          Organization

Case: CASE-003  |  Type: Commercial
Text: Organization Z appealed against the ruling by Judge E at the...
  Entity Found                   Label        What it means
  -------------------------------------------------------
  Organization Z                 ORG          Organization
  the Court of

In [8]:
# Hidden Markov Model

print("HIDDEN MARKOV MODEL (HMM)")
print("=" * 55)
print()
print("Example from a court sentence:")
print()

sample = "The court dismissed the case on Monday."
tokens  = word_tokenize(sample)
tags    = pos_tag(tokens)

tag_explain = {
    'DT' : 'Determiner',
    'NN' : 'Common Noun',
    'NNP': 'Proper Noun',
    'VBD': 'Past Verb',
    'IN' : 'Preposition',
    'NNS': 'Plural Noun',
    '.'  : 'Punctuation',
}

print(f"Sentence: {sample}")
print()
print(f"  {'Step':<6} {'Observed Word':<20} {'Hidden State (POS)':<20} {'Meaning'}")
print(f"  {'-'*65}")
for i, (word, tag) in enumerate(tags, 1):
    meaning = tag_explain.get(tag, tag)
    print(f"  {i:<6} {word:<20} {tag:<20} {meaning}")

print()
print("The HMM learns: after seeing 'The', next is likely a NOUN.")


HIDDEN MARKOV MODEL (HMM)

Example from a court sentence:

Sentence: The court dismissed the case on Monday.

  Step   Observed Word        Hidden State (POS)   Meaning
  -----------------------------------------------------------------
  1      The                  DT                   Determiner
  2      court                NN                   Common Noun
  3      dismissed            VBD                  Past Verb
  4      the                  DT                   Determiner
  5      case                 NN                   Common Noun
  6      on                   IN                   Preposition
  7      Monday               NNP                  Proper Noun
  8      .                    .                    Punctuation

The HMM learns: after seeing 'The', next is likely a NOUN.


In [9]:
# Named Entity Recognition

print("NAMED ENTITY RECOGNITION — SINGLE CASE EXAMPLE")
print("=" * 55)

example_text = "Person A filed a case against Organization X in City A on 10th March 2023. Judge B delivered the judgment."
doc = nlp(example_text)

print(f"Input Text: {example_text}")
print()
print(f"  {'Entity':<30} {'Type':<12} {'Meaning'}")
print(f"  {'-'*55}")
for ent in doc.ents:
    meaning = label_map.get(ent.label_, ent.label_)
    print(f"  {ent.text:<30} {ent.label_:<12} {meaning}")

NAMED ENTITY RECOGNITION — SINGLE CASE EXAMPLE
Input Text: Person A filed a case against Organization X in City A on 10th March 2023. Judge B delivered the judgment.

  Entity                         Type         Meaning
  -------------------------------------------------------
  Organization X in City A       ORG          Organization
  10th March 2023                DATE         Date


In [11]:
#Sentence Labeling Results

print("SENTENCE LABELING RESULTS — ALL COURT CASES")
print("=" * 65)

results = []
for doc_data in court_documents:
    doc = nlp(doc_data['text'])
    for ent in doc.ents:
        results.append({
            'Case ID'  : doc_data['case_id'],
            'Case Type': doc_data['type'],
            'Entity'   : ent.text,
            'Label'    : ent.label_,
            'Meaning'  : label_map.get(ent.label_, ent.label_)
        })

df = pd.DataFrame(results)
print(df.to_string(index=False))

print()
print("SUMMARY")
print("-" * 40)
print(f"Total entities extracted : {len(df)}")
print(f"Entity types found       : {df['Label'].unique().tolist()}")
print(f"Cases processed          : {df['Case ID'].nunique()}")
print()


SENTENCE LABELING RESULTS — ALL COURT CASES
 Case ID      Case Type                           Entity   Label         Meaning
CASE-001          Civil Organization X in the High Court     ORG    Organization
CASE-001          Civil                  14th March 2023    DATE            Date
CASE-002       Criminal                 5th January 2023    DATE            Date
CASE-002       Criminal                  Organization Y.     ORG    Organization
CASE-003     Commercial                   Organization Z     ORG    Organization
CASE-003     Commercial              the Court of Appeal     ORG    Organization
CASE-003     Commercial                  20th April 2023    DATE            Date
CASE-004           Land  The County Government of Region     ORG    Organization
CASE-004           Land                                F  PERSON     Person Name
CASE-004           Land                             Town     GPE Location / City
CASE-004           Land                8th February 2023    DATE 